# Phase 3 — Feature Extraction

This phase takes the cleaned corpus from Phase 1 and the labeled training set from Phase 2, and produces two types of features used in downstream modeling (Phase 4):

1. **Dense embeddings** — semantic vectors for every passage, used as the backbone of the kNN classifier
2. **Interpretability features** — lightweight rule-based signals (hedge density, scope, certifications) used to explain *why* a passage was flagged, not to classify it

### Pipeline Overview
**`passages.jsonl` + `labeled_passages.jsonl` → embeddings → rule-based features → `features.parquet`**

### Outputs
| File | Contents |
|---|---|
| `corpus_embeddings.npy` | Shape `(N, 384)` — all passages encoded, L2-normalised |
| `labeled_embeddings.npy` | Shape `(train_n, 384)` — training set encoded, L2-normalised |
| `features.parquet` | Per-passage interpretability features for all corpus passages |

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import textstat
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from codecarbon import EmissionsTracker

## Configuration & Data Loading

Loads two datasets:
- **`passages.jsonl`** — the full cleaned corpus (~4,500 passages after preprocessing and dedup); all passages get embeddings and interpretability features computed
- **`labeled_passages.jsonl`** — just the training split from annotation (~52 passages); gets a separate embedding matrix used as the kNN reference set in Phase 4

Both are loaded at the same time so the model only needs to be initialised once.

In [2]:
ROOT       = Path('..').resolve()
PASSAGES   = ROOT / 'data' / 'extracted' / 'passages.jsonl'
LABELED    = ROOT / 'data' / 'labeled' / 'labeled_passages.jsonl'
FEAT_DIR   = ROOT / 'data' / 'features'
FEAT_DIR.mkdir(parents=True, exist_ok=True)
CARBON_DIR = ROOT / 'results' / 'carbon'
CARBON_DIR.mkdir(parents=True, exist_ok=True)

with open(PASSAGES) as f:
    corpus = [json.loads(l) for l in f]
with open(LABELED) as f:
    labeled = [json.loads(l) for l in f]

corpus_texts  = [p['text'] for p in corpus]
labeled_texts = [p['text'] for p in labeled]
print(f"Corpus: {len(corpus_texts):,} passages | Labeled train: {len(labeled_texts)}")

Corpus: 4,485 passages | Labeled train: 92


In [3]:
tracker = EmissionsTracker(
    project_name="green-claims-nlp",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    log_level="error",
)
tracker.start()
print("Carbon tracking started.")

Carbon tracking started.


## Embeddings — `all-MiniLM-L6-v2`

**Why this model:** `all-MiniLM-L6-v2` is a distilled sentence transformer that produces 384-dimensional vectors. It's fast enough to run on CPU for ~4,500 passages and captures sentence-level semantics well — meaning passages with similar *meaning* (not just shared keywords) end up close together in vector space. This matters for greenwashing detection because the same vague claim can be phrased many different ways.

**Two embedding matrices are produced:**
- `corpus_embeddings` — every passage in the full corpus; used in Phase 4 to find nearest neighbours for any new passage
- `labeled_embeddings` — only the annotated training passages; used as the labelled reference set that kNN votes from

**L2 normalisation:** Both matrices are L2-normalised after encoding. This converts the inner product between any two vectors into their cosine similarity, which is the standard distance metric for kNN on text embeddings. Without normalisation, longer passages (which produce larger-magnitude vectors) would unfairly dominate similarity scores.

In [4]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

corpus_embeddings  = model.encode(corpus_texts,  batch_size=64, show_progress_bar=True, convert_to_numpy=True)
labeled_embeddings = model.encode(labeled_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

# L2-normalize for cosine similarity in kNN
corpus_embeddings  = normalize(corpus_embeddings,  norm='l2')
labeled_embeddings = normalize(labeled_embeddings, norm='l2')

np.save(FEAT_DIR / 'corpus_embeddings.npy',  corpus_embeddings)
np.save(FEAT_DIR / 'labeled_embeddings.npy', labeled_embeddings)
print(f"corpus_embeddings:  {corpus_embeddings.shape}")
print(f"labeled_embeddings: {labeled_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/71 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

corpus_embeddings:  (4485, 384)
labeled_embeddings: (92, 384)


## Supplementary Features

These are **not inputs to the kNN classifier** — they are rule-based signals computed for every corpus passage and saved to `features.parquet`. They are used in Phase 4 and Phase 5 to:
- Explain *why* a passage received a particular label (interpretability)
- Surface patterns across brands (e.g. which brands use the most vague language)
- Cross-reference model predictions against known divergence signals

### Feature Definitions

| Feature | Type | What it captures |
|---|---|---|
| `vague_language_score` | float | Ratio of vague commitment words (`GREENWASH_TERMS`) to total words. Based on Loughran & McDonald (2011), extended with domain terms. Higher = more hedging = greenwashing signal. |
| `redflag_standard` | 0/1 | 1 if the passage uses a self-defined standard (`REDFLAG_TERMS`: "sustainably sourced", "Conscious Collection", "Join Life") *without* citing a verified certification (`VERIFIED_CERTIFICATIONS`: SBTi, GRI, FSC, B Corp, etc.). |
| `has_specific_numbers` | 0/1 | 1 if the passage contains a percentage, a target year (2020–2049), or an emissions/energy unit (tonnes, kWh, MWh). Proxy for specificity — passages with no numbers are more likely to be vague aspirational language. |
| `scope` | string | Which part of the value chain the claim covers: `own_ops` (Scope 1/2, direct operations), `supply_chain` (Scope 3, suppliers), `mixed`, or `unclear`. Supply chain claims are harder to verify. |
| `readability_grade` | float | Flesch-Kincaid reading grade level. Very high scores (>20) often indicate dense legal boilerplate rather than narrative claims. |
| `unregulated_beauty_claim` | 0/1 | Clean beauty only. 1 if the passage uses unregulated marketing terms (`BEAUTY_REDFLAG_TERMS`: "clean", "natural", "non-toxic", "pure") that have no legal definition. Always 0 for fashion passages. |
| `text_length_chars` | int | Raw character count of the passage. |
| `has_table` | bool | Whether the passage came from a page containing a Markdown table (often emissions data). |

In [5]:
# ── Vague commitment language ─────────────────────────────────────────────────
# Words and phrases that sound positive but make no verifiable commitment.
# Base: Loughran & McDonald (2011) uncertainty wordlist (selected)
# Domain additions from pilot reading of H&M and Inditex reports
GREENWASH_TERMS = {
    # Loughran-McDonald uncertainty
    'approximately', 'uncertain', 'uncertainties', 'unclear', 'ambiguous',
    'depend', 'depends', 'depending', 'contingent', 'indefinite',
    # Vague commitments
    'committed to', 'working towards', 'aims to', 'strives to', 'exploring',
    'ambition is', 'journey towards', 'we believe', 'we strive',
    'in a sustainable way', 'positive impact', 'conscious', 'responsible',
    'hope to', 'intend to', 'plan to', 'seek to', 'aspire',
}

def vague_language_score(text: str) -> float:
    """Ratio of vague/hedge words to total words. Higher = more greenwashing signal."""
    text_lower = text.lower()
    hits = sum(1 for term in GREENWASH_TERMS if term in text_lower)
    words = len(text_lower.split())
    return hits / words if words > 0 else 0.0


# ── Verified certifications (exempt from red flag) ────────────────────────────
# If one of these is cited alongside a vague term, the passage is not flagged.
VERIFIED_CERTIFICATIONS = {
    'sbti', 'science based targets', 'gri', 'zdhc', 'fsc', 'bluesign',
    'oeko-tex', 'oeko tex', 'fair trade', 'fairtrade', 'b corp', 'bcorp',
    'deloitte', 'pwc', 'kpmg', 'ey assurance', 'isae', 'cdp',
    'better cotton initiative', 'rainforest alliance', 'rspo', 'gots',
}

# ── Red-flag terms: self-defined standards with no third-party verification ───
# Brands use these terms to imply rigour without citing any external certifier.
REDFLAG_TERMS = {
    'lower-impact', 'lower impact', 'sustainably sourced', 'responsibly made',
    'conscious choice', 'eco-designed', 'eco designed',
    'better cotton',    # flag unless BCI outcome metrics present
    'conscious',        # H&M "Conscious Collection"
    'join life',        # Inditex/Zara "Join Life" label
    'preferred fiber',  # common industry euphemism with no fixed standard
    'preferred material',
}

def redflag_standard(text: str) -> int:
    """1 if passage uses a self-defined standard without citing a verified certification."""
    text_lower = text.lower()
    has_redflag  = any(t in text_lower for t in REDFLAG_TERMS)
    has_verified = any(t in text_lower for t in VERIFIED_CERTIFICATIONS)
    return int(has_redflag and not has_verified)


# ── Specific numbers / measurable targets ────────────────────────────────────
# Proxy for claim specificity. Years are restricted to 2020–2050 to avoid
# matching financial figures, employee headcounts, style codes, etc.
_QUANT_RE = re.compile(
    r'(\d+\.?\d*\s*%'                          # percentage
    r'|20[2-4]\d'                               # year 2020–2049
    r'|\d+\.?\d*\s*(tonnes?|tco2e?|kwh|mwh|gwh|kg|mt))',
    re.IGNORECASE,
)

def has_specific_numbers(text: str) -> int:
    """1 if the passage contains a percentage, a target year, or an emissions/energy unit."""
    return int(bool(_QUANT_RE.search(text)))


# ── Scope of claim ────────────────────────────────────────────────────────────
# Own operations (Scope 1/2) are easier to verify than supply chain (Scope 3).
OWN_OPS_KW = {
    'our stores', 'our offices', 'scope 1', 'scope 2', 'own operations',
    'our facilities', 'our headquarters', 'our buildings', 'direct operations',
}
SUPPLY_KW = {
    'supply chain', 'suppliers', 'scope 3', 'value chain',
    'tier 1', 'tier 2', 'upstream', 'downstream', 'manufacturers',
    'sourcing', 'raw materials', 'vendor',
}

def scope_flag(text: str) -> str:
    """Classify which part of the value chain the claim covers."""
    text_lower = text.lower()
    own = any(k in text_lower for k in OWN_OPS_KW)
    sup = any(k in text_lower for k in SUPPLY_KW)
    if own and sup: return 'mixed'
    if own:         return 'own_ops'
    if sup:         return 'supply_chain'
    return 'unclear'


# ── Unregulated beauty marketing terms ───────────────────────────────────────
# These terms have no legal definition and cannot be independently verified.
# Only applied to clean_beauty industry passages.
BEAUTY_REDFLAG_TERMS = {
    'clean', 'natural', 'non-toxic', 'nontoxic', 'reef-safe', 'reef safe',
    'green', 'pure', 'eco', 'gentle', 'plant-based', 'plant based',
}

def unregulated_beauty_claim(text: str, industry: str) -> int:
    """1 if a clean beauty passage uses unregulated marketing terms."""
    if industry != 'clean_beauty':
        return 0
    text_lower = text.lower()
    return int(any(t in text_lower for t in BEAUTY_REDFLAG_TERMS))


print("Feature functions defined.")

Feature functions defined.


In [6]:
# Compute all features for the full corpus
rows = []
for p in corpus:
    rows.append({
        'passage_id':               p['passage_id'],
        'brand':                    p['brand'],
        'industry':                 p['industry'],
        'role':                     p['role'],
        'vague_language_score':     vague_language_score(p['text']),
        'redflag_standard':         redflag_standard(p['text']),
        'has_specific_numbers':     has_specific_numbers(p['text']),
        'scope':                    scope_flag(p['text']),
        'readability_grade':        textstat.flesch_kincaid_grade(p['text']),
        'unregulated_beauty_claim': unregulated_beauty_claim(p['text'], p['industry']),
        'text_length_chars':        len(p['text']),
        'has_table':                p.get('has_table', False),
    })

features_df = pd.DataFrame(rows)
features_df.to_parquet(FEAT_DIR / 'features.parquet', index=False)
print(f"Saved {len(features_df):,} rows → {FEAT_DIR / 'features.parquet'}")
features_df.describe()

Saved 4,485 rows → /Users/mandy.sun/green-claims-nlp/data/features/features.parquet


,passage_id,vague_language_score,redflag_standard,has_specific_numbers,readability_grade,unregulated_beauty_claim,text_length_chars
count,4485.000000,4485.000000,4485.000000,4485.000000,4485.000000,4485.000000,4485.000000
mean,2242.000000,0.002896,0.053512,0.678484,16.744133,0.107692,892.176143
std,1294.852308,0.005319,0.225077,0.467110,11.446759,0.310026,573.572274
min,0.000000,0.000000,0.000000,0.000000,5.096044,0.000000,89.000000
25%,1121.000000,0.000000,0.000000,0.000000,13.690643,0.000000,654.000000
50%,2242.000000,0.000000,0.000000,1.000000,15.851192,0.000000,798.000000
75%,3363.000000,0.005917,0.000000,1.000000,18.311743,0.000000,981.000000
max,4484.000000,0.066667,1.000000,1.000000,426.495216,1.000000,13788.000000


## Compute & Save Features

Applies all feature functions to every passage in the full corpus and saves the result as a Parquet file. Parquet is used instead of CSV because it preserves column types (bool, float, int, string) without any parsing ambiguity on reload.

The output includes `passage_id`, `brand`, `industry`, and `role` as join keys so `features.parquet` can be merged with embeddings or labels in downstream notebooks using `passage_id`.

In [7]:
emissions = tracker.stop()
print(f"Total feature extraction emissions: {emissions:.4f} kg CO2e")
print(f"Saved to: {CARBON_DIR / 'emissions.csv'}")

Total feature extraction emissions: 0.0001 kg CO2e
Saved to: /Users/mandy.sun/green-claims-nlp/results/carbon/emissions.csv
